# NCCT Stroke Segmentation — Architecture Search with MMSegmentation

Segment ischemic stroke regions from Non-Contrast CT (NCCT) brain images using the [MMSegmentation](https://github.com/open-mmlab/mmsegmentation) framework.

**This notebook:**
1. Clones the [ncct-segmentation](https://github.com/lhfazry/ncct-segmentation) repo (MMSegmentation fork)
2. Installs MMCV, MMEngine, and the project
3. Downloads the NCCT stroke dataset
4. Creates configs for **6 architectures** (U-Net variants, DeepLabV3+, SegFormer, PSPNet)
5. Trains all models with a quick schedule (~3K iterations each)
6. **Compares results** — mDice, mIoU tables + learning curves
7. Evaluates the best model on the test set

---
**Runtime**: T4 GPU or better recommended  
**Storage needed**: ~10GB

## 1. Environment Setup

### Verify GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Install Dependencies

MMSegmentation requires:
- **MMCV**: Pre-built CUDA wheel from MiroPsota's community wheel builder
- **MMEngine**: Via pip
- **Project**: Install the forked repo in editable mode

In [ ]:
import torch, sys, os, subprocess, time
torch_ver = torch.__version__.split("+")[0]
cuda_ver = torch.version.cuda
cuda_short = cuda_ver.replace(".", "")
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"torch={torch_ver}  cuda={cuda_ver}  python={sys.version_info.major}.{sys.version_info.minor}")

!pip install -q gdown

# Uninstall mmcv-lite if previously installed
!pip uninstall mmcv-lite -y -q 2>/dev/null || true

# ---- Install MMCV ----
RELEASE_TAG = "mmcv-2.2.0%2Ba8073c7"
WHEEL_FILE = f"mmcv-2.2.0%2Ba8073c7pt{torch_ver}cu{cuda_short}-{py_ver}-{py_ver}-linux_x86_64.whl"
WHEEL_URL = f"https://github.com/MiroPsota/torch_packages_builder/releases/download/{RELEASE_TAG}/{WHEEL_FILE}"
WHEEL_PATH = f"/tmp/{WHEEL_FILE}"
print(f"Wheel: pt{torch_ver}cu{cuda_short} {py_ver}")

MMCV_INSTALLED = False

if not os.path.exists(WHEEL_PATH):
    ret = os.system(f"curl -L --retry 5 --retry-delay 10 --connect-timeout 30 "
                    f"'{WHEEL_URL}' -o '{WHEEL_PATH}' 2>&1 | tail -3")
else:
    ret = 0
    print("Wheel already downloaded.")

if ret == 0 and os.path.exists(WHEEL_PATH) and os.path.getsize(WHEEL_PATH) > 1_000_000:
    print("Installing MMCV from downloaded wheel...")
    r = subprocess.run(["pip", "install", WHEEL_PATH], capture_output=True, text=True)
    if r.returncode == 0:
        print("MMCV installed from community wheel.")
        MMCV_INSTALLED = True
    else:
        print(f"pip install failed: {r.stderr[-200:]}")
        os.remove(WHEEL_PATH)

if not MMCV_INSTALLED:
    print("Community wheel unavailable. Building MMCV from source via pip...")
    print("This takes 10-15 minutes on Colab.")
    !pip install "mmcv>=2.0.0rc4,<2.5.0" -q
    print("MMCV installed via pip.")

!pip install -q mmengine

import mmcv
print(f"mmcv: {mmcv.__version__}")
print(f"mmcv.ops available: {hasattr(mmcv, 'ops')}")

In [ ]:
import os
from pathlib import Path

# Clone the repository
REPO_URL = "https://github.com/lhfazry/ncct-segmentation"
REPO_DIR = "/content/ncct-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git status --short

In [ ]:
# Install the project in editable mode
!pip install -e . -q
!pip install ftfy regex albumentations pandas -q

# Patch mmseg/__init__.py: relax MMCV version check
import re
with open("/content/ncct-segmentation/mmseg/__init__.py") as f:
    _init_code = f.read()
_init_code = _init_code.replace(
    "MMCV_MAX = '2.3.0'",
    "MMCV_MAX = '3.0.0'")
_init_code = re.sub(
    r'assert \(mmcv_min_version <= mmcv_version < mmcv_max_version\),.*?'
    r'f\'Please install mmcv>=2\.0\.0rc4\.\'',
    """if not (mmcv_min_version <= mmcv_version < mmcv_max_version):
    import warnings
    warnings.warn(
        f'MMCV=={mmcv.__version__} is used but may be incompatible. '
        f'Expected mmcv>={MMCV_MIN}, <{MMCV_MAX}.')""",
    _init_code,
    flags=re.DOTALL,
)
with open("/content/ncct-segmentation/mmseg/__init__.py", "w") as f:
    f.write(_init_code)
print("Patched mmseg/__init__.py: MMCV version assert \u2192 warning")

# Patch mmseg/models/backbones/mit.py: fix Version vs tuple comparison
# mmcv_version is a packaging.version.Version object, but digit_version returns
# a tuple — Python 3 raises TypeError comparing them.
mit_path = "/content/ncct-segmentation/mmseg/models/backbones/mit.py"
with open(mit_path) as f:
    mit_code = f.read()
mit_code = mit_code.replace(
    "import math\nimport warnings",
    "import math\nimport warnings\n\nfrom packaging.version import Version"
)
mit_code = mit_code.replace(
    "if mmcv_version < digit_version('1.3.17'):",
    "if mmcv_version < Version('1.3.17'):"
)
with open(mit_path, "w") as f:
    f.write(mit_code)
print("Patched mmseg/models/backbones/mit.py: Version comparison fix")

import mmseg
from mmseg.utils import register_all_modules
register_all_modules()
print(f"mmseg version: {mmseg.__version__}")

---
## 2. Dataset Preparation

Download the NCCT brain dataset, extract it, and organize for MMSegmentation.

In [ ]:
import gdown
import zipfile
import shutil

# Google Drive file ID
FILE_ID = "1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo"
ZIP_PATH = "/content/dataset.zip"
EXTRACT_DIR = "/content/dataset_raw"

if not os.path.exists(ZIP_PATH):
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    print("Downloading dataset from Google Drive...")
    gdown.download(url, ZIP_PATH, quiet=False)
else:
    print("Dataset zip already downloaded.")

In [ ]:
# Extract dataset
if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
# Organize into MMSegmentation-compatible structure
# data/ncct/
#   train/images/  -> *.png
#   train/masks/   -> *.png
#   val/images/    -> *.png
#   val/masks/     -> *.png
#   test/images/   -> *.png
#   test/masks/    -> *.png

DATA_ROOT = "/content/data/ncct"
os.makedirs(DATA_ROOT, exist_ok=True)

extracted_contents = os.listdir(EXTRACT_DIR)
print(f"Extracted contents: {extracted_contents}")

expected_split_dirs = ["train", "val", "test"]
found_splits = [d for d in expected_split_dirs if os.path.isdir(os.path.join(EXTRACT_DIR, d))]

if found_splits:
    print(f"Found splits: {found_splits}")
    for split in found_splits:
        src_img = os.path.join(EXTRACT_DIR, split, "images")
        src_mask = os.path.join(EXTRACT_DIR, split, "masks")
        if os.path.isdir(src_img) and os.path.isdir(src_mask):
            dst_img = os.path.join(DATA_ROOT, split, "images")
            dst_mask = os.path.join(DATA_ROOT, split, "masks")
            os.makedirs(os.path.dirname(dst_img), exist_ok=True)
            if not os.path.exists(dst_img):
                shutil.copytree(src_img, dst_img)
            if not os.path.exists(dst_mask):
                shutil.copytree(src_mask, dst_mask)

print(f"\nDataset organized at: {DATA_ROOT}")

In [ ]:
# Verify dataset structure
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATA_ROOT, split, "images")
    mask_dir = os.path.join(DATA_ROOT, split, "masks")
    if os.path.isdir(img_dir) and os.path.isdir(mask_dir):
        images = sorted(os.listdir(img_dir))
        masks = sorted(os.listdir(mask_dir))
        print(f"{split:5s}: {len(images):4d} images, {len(masks):4d} masks")
        if images and masks:
            print(f"         Sample: {images[0]}")
    else:
        print(f"{split:5s}: NOT FOUND")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Quick visualization of dataset samples
img_dir = os.path.join(DATA_ROOT, "train", "images")
mask_dir = os.path.join(DATA_ROOT, "train", "masks")

if os.path.isdir(img_dir):
    imgs = sorted(os.listdir(img_dir))[:3]
    fig, axes = plt.subplots(len(imgs), 2, figsize=(8, 3*len(imgs)))
    for i, fname in enumerate(imgs):
        img = Image.open(os.path.join(img_dir, fname)).convert("L")
        mask = Image.open(os.path.join(mask_dir, fname)).convert("L")

        axes[i][0].imshow(img, cmap="gray")
        axes[i][0].set_title(f"Input: {fname}")
        axes[i][0].axis("off")

        axes[i][1].imshow(mask, cmap="gray")
        axes[i][1].set_title(f"Mask (stroke in white)")
        axes[i][1].axis("off")

    plt.tight_layout()
    plt.show()

    # Show class distribution
    total_pixels = 0
    stroke_pixels = 0
    for fname in os.listdir(mask_dir)[:100]:
        m = np.array(Image.open(os.path.join(mask_dir, fname)).convert("L"))
        total_pixels += m.size
        stroke_pixels += (m > 127).sum()
    print(f"Class balance (first 100 training samples):")
    print(f"  Background: {(total_pixels - stroke_pixels) / total_pixels * 100:.2f}%")
    print(f"  Stroke:     {stroke_pixels / total_pixels * 100:.2f}%")

---
## 3. Multi-Architecture Configuration

We'll create configs for **6 architectures** covering different paradigms:

| # | Model | Backbone | Head | Paradigm |
|---|-------|----------|------|----------|
| 1 | **U-Net (FCN)** | U-Net (s5-d16) | FCNHead | CNN encoder-decoder (baseline) |
| 2 | **U-Net + DeepLabV3** | U-Net (s5-d16) | ASPPHead | U-Net + atrous spatial pyramid |
| 3 | **U-Net + PSPNet** | U-Net (s5-d16) | PSPHead | U-Net + pyramid pooling |
| 4 | **DeepLabV3+** | ResNet-50 | DeepSepASPPHead | ResNet + ASPP + decoder |
| 5 | **SegFormer** | MixVisionTransformer (B0) | SegformerHead | Lightweight transformer |
| 6 | **PSPNet** | ResNet-50 | PSPHead | Pyramid scene parsing |

All use the same training schedule (3K iterations), data pipeline, loss, and class weights for fair comparison.

In [ ]:
from mmengine.config import Config
import os

NCCT_CFG_DIR = "/content/ncct-segmentation/configs/ncct"
COLAB_CFG_DIR = "/tmp/colab_configs"
os.makedirs(COLAB_CFG_DIR, exist_ok=True)

MAX_ITERS = 1000
VAL_INTERVAL = 500

# All 6 model configs are already in the repo under configs/ncct/
# We just need to load and patch data_root for the Colab environment.
ARCHES = [
    ("unet_fcn",           "unet_fcn_stroke_ncct.py"),
    ("unet_deeplabv3",     "unet_deeplabv3_stroke_ncct.py"),
    ("unet_pspnet",        "unet_pspnet_stroke_ncct.py"),
    ("deeplabv3plus_r50",  "deeplabv3plus_r50_stroke_ncct.py"),
    ("segformer_mitb0",    "segformer_mitb0_stroke_ncct.py"),
    ("pspnet_r50",         "pspnet_r50_stroke_ncct.py"),
]

configs = {}
for name, cfg_file in ARCHES:
    cfg_path = os.path.join(NCCT_CFG_DIR, cfg_file)
    cfg = Config.fromfile(cfg_path)
    # Patch data_root for Colab (dataset is at /content/data/ncct/)
    # IMPORTANT: dataset dicts capture data_root BY VALUE at config load time.
    # Changing cfg.data_root alone does NOT update nested dicts like
    # cfg.train_dataloader.dataset.data_root — they must be set explicitly.
    _abs_root = DATA_ROOT + "/"
    cfg.data_root = _abs_root
    for _dl_key in ["train_dataloader", "val_dataloader", "test_dataloader"]:
        if hasattr(cfg, _dl_key) and hasattr(getattr(cfg, _dl_key), "dataset"):
            getattr(cfg, _dl_key).dataset.data_root = _abs_root
    # Set work directory for this run
    cfg.work_dir = f"/content/work_dirs/{name}"
    # Dump patched config so tools/train.py uses the correct paths
    colab_path = os.path.join(COLAB_CFG_DIR, cfg_file)
    cfg.dump(colab_path)
    configs[name] = {"cfg": cfg, "path": colab_path}
    print(f"[OK] {name:20s} -> {colab_path}")

print(f"\nAll {len(ARCHES)} configs loaded.")


---
## 4. Training All Architectures

Each model is trained in its **own cell** below. Run them sequentially.
Output streams directly so you can monitor training progress in real time.

| # | Model | Config |
|---|-------|--------|
| 1 | **U-Net (FCN)** | `unet_fcn_stroke_ncct.py` |
| 2 | **U-Net + DeepLabV3** | `unet_deeplabv3_stroke_ncct.py` |
| 3 | **U-Net + PSPNet** | `unet_pspnet_stroke_ncct.py` |
| 4 | **DeepLabV3+ (R-50)** | `deeplabv3plus_r50_stroke_ncct.py` |
| 5 | **SegFormer (MIT-B0)** | `segformer_mitb0_stroke_ncct.py` |
| 6 | **PSPNet (R-50)** | `pspnet_r50_stroke_ncct.py` |

Each runs for **1000 iterations** with AMP (mixed precision) on a single GPU.
Expected runtime: ~30-60 min per model on T4.

> To train a subset, skip cells you don't need. Training status is auto-detected.


### **4.1. Train U-Net (FCN)**

In [ ]:
%%time
import os

_arch = "unet_fcn"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **4.2. Train U-Net + DeepLabV3**

In [ ]:
%%time
import os

_arch = "unet_deeplabv3"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **4.3. U-Net + PSPNet**

In [ ]:
%%time
import os

_arch = "unet_pspnet"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **4.4. Train DeepLabV3+ (R-50)**

In [ ]:
%%time
import os

_arch = "deeplabv3plus_r50"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **4.5 Train SegFormer (MIT-B0)**

In [ ]:
%%time
import os

_arch = "segformer_mitb0"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **4.6. Train PSPNet (R-50)**

In [ ]:
%%time
import os

_arch = "pspnet_r50"
_cfg_path = configs[_arch]["path"]
_work_dir = configs[_arch]["cfg"].work_dir

print("=" * 60)
print(f"  Training: {_arch}")
print(f"  Config:   {_cfg_path}")
print(f"  Work dir: {_work_dir}")
print(f"  Iters:    {MAX_ITERS}")
print("=" * 60)

os.makedirs(_work_dir, exist_ok=True)
!cd /content/ncct-segmentation && python -u tools/train.py {_cfg_path} \
    --work-dir {_work_dir} --amp \
    --cfg-options train_cfg.max_iters={MAX_ITERS} data_root={DATA_ROOT}/

print(f"\nDone: {_arch}")
del _arch, _cfg_path, _work_dir


### **Collecting Training Results**

In [ ]:
# Collect training results from all work_dirs
import glob
import json

train_results = {}
for _name, _info in configs.items():
    _work_dir = _info["cfg"].work_dir
    _log_files = sorted(glob.glob(os.path.join(_work_dir, "*.log.json")))

    # Estimate elapsed time from log timestamps
    _elapsed = 0
    if _log_files:
        try:
            with open(_log_files[-1]) as _f:
                _lines = _f.readlines()
                if _lines:
                    _first = json.loads(_lines[0])
                    _last = json.loads(_lines[-1].strip())
                    _elapsed = _last.get("time", 0) - _first.get("time", 0)
        except Exception:
            pass

    train_results[_name] = {
        "work_dir": _work_dir,
        "elapsed": _elapsed,
    }

    _status = chr(10003) if _log_files else chr(10007)
    _elapsed_str = f"{_elapsed/60:.1f}m" if _elapsed else "N/A"
    print(f"[{_status}] {_name:20s} | elapsed: {_elapsed_str}")

print(f"\nCollected results for {len(train_results)} models.")
print("Proceed to the comparison section below.")


### Collecting Validation Metrics

Parse each model's JSON log to extract final validation mDice and mIoU.

In [ ]:
import json
import glob

def extract_val_metrics(work_dir):
    """Parse the JSON log file for validation metrics."""
    log_files = sorted(glob.glob(os.path.join(work_dir, "*.log.json")))
    if not log_files:
        return None

    val_metrics = []
    with open(log_files[-1]) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            if entry.get("mode") == "val" and "mDice" in entry:
                val_metrics.append({
                    "iter": entry.get("iteration", 0),
                    "mDice": entry.get("mDice", 0),
                    "mIoU": entry.get("mIoU", 0),
                    "aAcc": entry.get("aAcc", 0),
                })
    return val_metrics


# Collect metrics from all models
all_metrics = {}
for name, info in train_results.items():
    metrics = extract_val_metrics(info["work_dir"])
    all_metrics[name] = metrics
    if metrics:
        best = max(metrics, key=lambda x: x["mDice"])
        print(f"{name:20s} | Best mDice: {best['mDice']:.4f} | mIoU: {best['mIoU']:.4f} | iter {best['iter']}")
    else:
        print(f"{name:20s} | No validation metrics found (training may be incomplete)")

---
## 5. Architecture Comparison

Side-by-side comparison of all architectures. Models ranked by best validation mDice.

In [ ]:
import pandas as pd

if not all_metrics:
    print("No metrics collected. Make sure training ran before this cell.")
else:
    rows = []
    for name, metrics in all_metrics.items():
        if not metrics:
            continue
        best = max(metrics, key=lambda x: x["mDice"])
        final = metrics[-1]
        rows.append({
            "Model": name,
            "Best mDice": f"{best['mDice']:.4f}",
            "Best mIoU": f"{best['mIoU']:.4f}",
            "@iter": best["iter"],
            "Final mDice": f"{final['mDice']:.4f}",
            "Final mIoU": f"{final['mIoU']:.4f}",
            "Train time (min)": f"{train_results[name]['elapsed']/60:.1f}",
        })

    if not rows:
        print("No validation metrics found for any model. Training may be incomplete.")
    else:
        df = pd.DataFrame(rows)
        df = df.sort_values("Best mDice", ascending=False).reset_index(drop=True)
        print("Architecture Comparison (sorted by Best mDice):")
        print(df.to_string(index=False))

In [ ]:
# Plot validation curves for all models
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.tab10(np.linspace(0, 1, len(all_metrics)))

for (name, metrics), color in zip(all_metrics.items(), colors):
    if not metrics:
        continue
    iters = [m["iter"] for m in metrics]
    dices = [m["mDice"] for m in metrics]
    ious = [m["mIoU"] for m in metrics]

    # Shorten name for legend
    short = name.replace("unet_", "U-").replace("_r50", "").replace("_mitb0", "")
    axes[0].plot(iters, dices, "-o", color=color, label=short)
    axes[1].plot(iters, ious, "-s", color=color, label=short)

axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("mDice")
axes[0].set_title("Validation mDice")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("mIoU")
axes[1].set_title("Validation mIoU")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Best Model

Based on the comparison, the best architecture will be evaluated on the test set.

In [ ]:
# Select best model (highest final mDice)
best_model = None
best_score = -1
for name, metrics in all_metrics.items():
    if not metrics:
        continue
    final = metrics[-1]["mDice"]
    if final > best_score:
        best_score = final
        best_model = name

if best_model is None:
    print("No model with validation metrics found. Cannot select best model.")
    BEST_CKPT = None
else:
    print(f"Best architecture: {best_model}")
    print(f"Best validation mDice: {best_score:.4f}")

    BEST_CFG_PATH = configs[best_model]["path"]
    BEST_WORK_DIR = configs[best_model]["cfg"].work_dir

    # Find the latest checkpoint
    ckpts = sorted(glob.glob(os.path.join(BEST_WORK_DIR, "iter_*.pth")))
    if not ckpts:
        ckpts = sorted(glob.glob(os.path.join(BEST_WORK_DIR, "*.pth")))
    BEST_CKPT = ckpts[-1] if ckpts else None
    if BEST_CKPT:
        print(f"Checkpoint: {os.path.basename(BEST_CKPT)}")
    else:
        print("No checkpoint found!")

---
## 6. Best Model — Test Evaluation

Run the best model on the held-out test set for final metrics.

In [ ]:
%%time
if BEST_CKPT:
    !cd /content/ncct-segmentation && python tools/test.py \
        {BEST_CFG_PATH} \
        {BEST_CKPT} \
        --show-dir {BEST_WORK_DIR}/preds \
        --out {BEST_WORK_DIR}/results.pkl

In [ ]:
# Parse and display test metrics
if BEST_CKPT is None:
    print("No checkpoint available. Run training first.")
else:
    result = subprocess.run(
        ["python", "tools/test.py", BEST_CFG_PATH, BEST_CKPT,
         "--out", f"{BEST_WORK_DIR}/results.pkl"],
        capture_output=True, text=True, cwd="/content/ncct-segmentation"
    )

    print(f"Test Results for: {best_model}")
    print("=" * 40)
    for line in result.stdout.split("\n"):
        if any(kw in line for kw in ["Dice", "IoU", "mDice", "mIoU", "aAcc"]):
            print(line)
        if "OrderedDict" in line and ("Dice" in line or "IoU" in line):
            print(line)

    if not any(kw in result.stdout for kw in ["Dice", "mDice"]):
        for line in result.stderr.split("\n"):
            if any(kw in line for kw in ["Dice", "IoU", "mDice", "mIoU", "aAcc"]):
                print(line)

### Visualize Best Model Predictions

In [ ]:
from mmseg.apis import init_model, inference_model
from scipy.ndimage import binary_dilation

if BEST_CKPT is None:
    print("No checkpoint available. Run training first.")
else:
    # Load the best model
    model = init_model(BEST_CFG_PATH, BEST_CKPT, device="cuda:0")

    # Get test images
    test_img_dir = os.path.join(DATA_ROOT, "test/images")
    test_mask_dir = os.path.join(DATA_ROOT, "test/masks")
    test_images = sorted(os.listdir(test_img_dir))[:6]

    fig, axes = plt.subplots(len(test_images), 4, figsize=(16, 4 * len(test_images)))

    for i, fname in enumerate(test_images):
        img_path = os.path.join(test_img_dir, fname)
        mask_path = os.path.join(test_mask_dir, fname)

        # Load ground truth
        img_np = np.array(Image.open(img_path).convert("L"))
        mask_np = np.array(Image.open(mask_path).convert("L"))
        mask_bin = (mask_np > 127).astype(np.uint8)

        # Run inference
        result = inference_model(model, img_path)
        pred = result.pred_sem_seg.data.cpu().numpy()

        # Plot
        axes[i][0].imshow(img_np, cmap="gray")
        axes[i][0].set_title("Input NCCT")
        axes[i][0].axis("off")

        axes[i][1].imshow(mask_bin, cmap="gray")
        axes[i][1].set_title("Ground Truth")
        axes[i][1].axis("off")

        axes[i][2].imshow(pred, cmap="gray")
        axes[i][2].set_title("Prediction")
        axes[i][2].axis("off")

        # Overlay: prediction outline on input
        outline = binary_dilation(pred, iterations=1) ^ pred
        overlay = np.stack([img_np] * 3, axis=-1).astype(np.float32)
        overlay[:, :, 0] = np.where(outline, 255, overlay[:, :, 0])
        overlay[:, :, 1] = np.where(outline, 0, overlay[:, :, 1])
        overlay[:, :, 2] = np.where(outline, 0, overlay[:, :, 2])
        axes[i][3].imshow(overlay.astype(np.uint8))
        axes[i][3].set_title("Overlay (Pred edge)")
        axes[i][3].axis("off")

    plt.tight_layout()
    plt.show()

---
## 7. Save and Export

Save the best model weights and results to Google Drive.

In [ ]:
from google.colab import drive

# Mount Google Drive
DRIVE_MOUNT = "/content/drive"
drive.mount(DRIVE_MOUNT)

# Copy best checkpoint and results to Drive
import shutil
DRIVE_DST = "/content/drive/MyDrive/ncct_segmentation_results"
os.makedirs(DRIVE_DST, exist_ok=True)

if BEST_CKPT is None:
    print("No checkpoint to save. Run training first.")
else:
    # Copy checkpoint
    ckpt_name = os.path.basename(BEST_CKPT)
    shutil.copy2(BEST_CKPT, os.path.join(DRIVE_DST, f"{best_model}_{ckpt_name}"))
    print(f"Copied {ckpt_name} to Drive")

    # Copy config
    shutil.copy2(BEST_CFG_PATH, os.path.join(DRIVE_DST, f"{best_model}_stroke_ncct.py"))
    print("Copied best config to Drive")

    # Copy comparison table
    try:
        df.to_csv(os.path.join(DRIVE_DST, "architecture_comparison.csv"), index=False)
        print("Copied comparison table to Drive")
    except NameError:
        print("Comparison table not available, skipping.")

    # Save summary as text
    summary_path = os.path.join(DRIVE_DST, "results_summary.txt")
    with open(summary_path, "w") as f:
        f.write("NCCT Stroke Segmentation - Architecture Search Results\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Schedule: {MAX_ITERS} iterations per model\n\n")
        try:
            f.write(df.to_string() + "\n\n")
        except NameError:
            pass
        f.write(f"Best model: {best_model} (val mDice: {best_score:.4f})\n")
    print(f"Saved summary to {summary_path}")

    # Copy results.pkl
    results_pkl = f"{BEST_WORK_DIR}/results.pkl"
    if os.path.exists(results_pkl):
        shutil.copy2(results_pkl, os.path.join(DRIVE_DST, f"{best_model}_results.pkl"))
        print("Copied test results to Drive")

print(f"\nAll files saved to: {DRIVE_DST}")

---
## 8. Retrain Best Model (Full Schedule)

Once you've identified the best architecture, re-run it with the **full 20K schedule**:

```python
BEST_MODEL = "<name_from_comparison>"
BEST_CFG = f"/content/ncct-segmentation/configs/ncct/{BEST_MODEL}_stroke_ncct.py"
FULL_WORK_DIR = f"/content/work_dirs/{BEST_MODEL}_full"
!cd /content/ncct-segmentation && python tools/train.py {BEST_CFG} \
    --work-dir {FULL_WORK_DIR} --amp \
    --cfg-options train_cfg.max_iters=20000 val_interval=2000
```

---
## Summary

1. **Data prepared**: NCCT dataset organized for MMSegmentation
2. **Configs created**: 6 architectures with identical training setup
3. **Models trained**: Quick 3K iteration schedule for fair comparison
4. **Best model selected**: Highest validation mDice
5. **Test evaluation**: Full metrics on held-out test set
6. **Results saved**: Checkpoint, config, comparison table to Drive

**Next step**: Train the best architecture with full 20K+ iterations for production use.